In [ ]:
exec(open(__import__('pathlib').Path(__vsc_ipynb_file__).parent.parent / 'src' / 'display_html.py').read())

## T_TaskEmbedding / T_FeatureEmbedding — K&M Rule-Action Task

**Role**: Kikumoto & Mayr (2020) rule-action selection task — `design_to_units`, `units_to_rsa`,
`T_FeatureEmbedding`, `T_TaskEmbedding` from `src/tasks.py`.
4 rules × 4 stimuli = **16 action contexts**; each context has a unique correct response.
`T_TaskEmbedding` converts condition indices (0–15) into rate-coded ECin patterns
whose similarity structure reflects the task feature hierarchy (STIM, RULE, RESP, SRCONJ, RSRCONJ).
ECin feeds directly into **MSP** (ECin→CA1, lr=0.05) and **TSP** (ECin→DG→CA3→CA1, lr=0.4).

**Representational hierarchy (Kikumoto et al. 2025 Fig. 1C)**:
- STIM: stimulus identity; 4 conditions per value; block-diagonal RSA
- RULE: rule identity; 4 conditions per value; stripe RSA
- RESP: response identity; 4 conditions per value
- SRCONJ: S-R conjunction; 2 conditions per pair; shared across rules (within=0.5)
- RSRCONJ: rule-specific S-R conjunction; 16 unique codes; **CA1 target** after CHL training

**Relevance to EChipp_SL**: One-hot ECin (one unit per condition) gives equidistant inputs —
MSP Hebbian CHL has no input similarity to amplify into conjunctive representations.
Feature-coded ECin (RULE + STIM blocks) seeds the similarity structure so CHL can develop
RSRCONJ conjunctions in CA1. `T_TaskEmbedding` with RSA initialization matches input geometry
to Kikumoto 2025 EEG at trial onset, making the input geometry a controllable simulation parameter.

**Understanding check**: Why does MSP need rate-coded (not one-hot per condition) ECin?
→ One-hot gives equidistant inputs; CHL updates ΔW = ActP⊗ActP − ActM⊗ActM,
so if all ECin patterns are equally dissimilar, CA1 cannot develop graded RSRCONJ geometry.
Why eigendecomposition for `rsa_to_embedding_init` rather than random init?
→ Eigenvectors of the RSA matrix give coordinates where cosine similarities reproduce the RSA target.

In [ ]:
# path & directories
import sys
from pathlib import Path

DIR_SRC  = str((Path(__vsc_ipynb_file__).parent.parent / 'src').resolve())
DIR_VIZ  = (Path(__vsc_ipynb_file__).parent.parent / 'visualizations').resolve()
TASKDIR  = (Path(__vsc_ipynb_file__).parent.parent / 'src' / 'z_task_design_tables' / 'RuleAction_4rules').resolve()
BASICDIR = TASKDIR / 'BASIC(4rules)'
sys.path.insert(0, DIR_SRC)

# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.decomposition import PCA
from tasks import design_to_units, units_to_rsa, T_FeatureEmbedding, T_TaskEmbedding

np.random.seed(42)
torch.manual_seed(42)

print(f'DIR_SRC: {DIR_SRC}')
print(f'TASKDIR: {TASKDIR}')
print(f'PyTorch: {torch.__version__}')

## Task Design

4 rules × 4 stimuli = 16 conditions. Each condition has a unique rule-specific S-R conjunction (RSRCONJ).

**Kikumoto et al. (2025) Fig. 1B:**  
- STIM (1–4): which dot position is presented  
- RULE (1–4): which spatial transformation rule is active (signaled by cue word)  
- RESP (1–4): which button is the correct response  
- SRCONJ (1–8): S-R conjunction shared across rules  
- RSRCONJ (1–16): rule-specific S-R conjunction — unique to each context

In [ ]:
design = pd.read_csv(TASKDIR / 'Task_Design.txt', sep='\t')
print(design.to_string(index=False))
print(f'\nShape: {design.shape}  ({design.shape[0]} conditions × {design.shape[1]} columns)')

# Verify expected counts
print(f'\nUnique values per column:')
for col in design.columns:
    print(f'  {col}: {sorted(design[col].unique())}')

In [ ]:
# One-hot unit matrices
units = design_to_units(design)

print('Unit matrix shapes (n_cond × n_values):')
for col, oh in units.items():
    print(f'  {col}: {oh.shape}')

# Verify one-hot correctness: each row should sum to 1
print('\nRow sums (should all be 1.0):')
for col, oh in units.items():
    sums = oh.sum(axis=1)
    ok = np.allclose(sums, 1.0)
    print(f'  {col}: {"OK" if ok else "FAIL"} (min={sums.min():.2f}, max={sums.max():.2f})')

## RSA Similarity Matrices (Computed)

Computed via `units_to_rsa()`: for each feature, `M[i,j] = dot(oh_norm[i], oh_norm[j])`  
where `oh_norm[i] = oh[i] / sqrt(row_sum)`.  
Because each condition has exactly one feature value (row_sum=1), this gives `M[i,j] ∈ {0, 1}`.

**Expected structure:**
- STIM: block-diagonal (4 blocks × 4; same stimulus across all rules)
- RULE: stripe-diagonal (4 stripes × 4; same rule across all stimuli)
- RESP: grouped by response (4 groups × 4, with complex pattern)
- SRCONJ: 8 pairs sharing S-R mapping across 2 rules each
- RSRCONJ: all 16 codes unique → identity matrix (1 only on diagonal)

In [ ]:
rsa = units_to_rsa(units)

cols_to_show = ['STIM', 'RULE', 'RESP', 'SRCONJ', 'RSRCONJ']
titles = [
    'STIM\n(stimulus identity)',
    'RULE\n(rule identity)',
    'RESP\n(response identity)',
    'SRCONJ\n(S-R conjunction)',
    'RSRCONJ\n(rule-specific S-R)',
]

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
for ax, col, title in zip(axes, cols_to_show, titles):
    if col not in rsa:
        ax.set_visible(False)
        continue
    sns.heatmap(rsa[col], ax=ax, vmin=0, vmax=1,
                cmap='Blues', linewidths=0.3, linecolor='lightgray',
                cbar=False, square=True,
                xticklabels=4, yticklabels=4)
    ax.set_title(title, fontsize=10)
    ax.tick_params(labelsize=7)

fig.suptitle('RSA Similarity Matrices — Computed via units_to_rsa()', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(DIR_VIZ / 'rsa_computed_4rules.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nRSRCONJ matrix (should be identity — all 16 codes unique):')
print(f'  Is identity: {np.allclose(rsa["RSRCONJ"], np.eye(16))}')

## Pre-loaded RSA Model Matrices

The `.txt` files in `z_task_design_tables/RuleAction_4rules/BASIC(4rules)/` contain  
pre-computed RSA matrices with a different normalization:  
each condition's one-hot is divided by **sqrt(group_size)**, not sqrt(row_sum=1).  

Result: `M[i,j] = 1/group_size` for same-group pairs (e.g., 0.25 for 4-member groups, 0.5 for 2-member groups).  
This makes within-group similarity scale consistently regardless of group size.

These are the **target RSA values** for comparing model CA1 representations to Kikumoto et al. (2025).

In [ ]:
# Load pre-computed RSA matrices from txt files
rsa_files = {
    'STIM'   : 'STIM.txt',
    'RULE'   : 'RULE.txt',
    'RESP'   : 'RESP.txt',
    'SRCONJ' : 'SRCONJ.txt',
    'RSRCONJ': 'RSRCONJ.txt',
}

rsa_loaded = {}
for col, fname in rsa_files.items():
    fpath = BASICDIR / fname
    if fpath.exists():
        rsa_loaded[col] = np.loadtxt(fpath, dtype=np.float32)
        print(f'{col}: shape={rsa_loaded[col].shape}, '
              f'unique values={np.unique(np.round(rsa_loaded[col], 4))}')
    else:
        print(f'{col}: file not found ({fpath})')

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
for ax, (col, title) in zip(axes, zip(cols_to_show, titles)):
    if col not in rsa_loaded:
        ax.set_visible(False)
        continue
    sns.heatmap(rsa_loaded[col], ax=ax, vmin=0, vmax=0.5,
                cmap='Oranges', linewidths=0.3, linecolor='lightgray',
                cbar=False, square=True,
                xticklabels=4, yticklabels=4)
    ax.set_title(title, fontsize=10)
    ax.tick_params(labelsize=7)

fig.suptitle('RSA Model Matrices — Pre-loaded from z_task_design_tables (group-size normalization)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(DIR_VIZ / 'rsa_preloaded_4rules.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nRSRCONJ matrix (pre-loaded) — first 4 rows:')
print(rsa_loaded['RSRCONJ'][:4])

## T_TaskEmbedding — RSA-Initialized Input Representations

`T_TaskEmbedding` converts condition index (0–15) into a rate-coded vector.  
With `emb_similarity=rsa_loaded`, weights are initialized via `rsa_to_embedding_init`:  
eigendecomposition of the RSA matrix → top-k eigenvectors scaled by √eigenvalue.  

**Why RSA initialization?**  
Random init gives equidistant embeddings — MSP has no signal to learn RSRCONJ structure.  
RSA init seeds ECin with STIM and RULE similarity, so CHL in CA1 can develop conjunctions.

**Configuration:**  
- `emb_dim = 8` per feature: RULE (8) + STIM (8) + RESP (8) + RSRCONJ (8) = **32 total**  
- `compositional_on = False` → RSRCONJ gets its own lookup table (independent of base features)

In [ ]:
EMB_DIM = 8

emb = T_TaskEmbedding(
    design=design,
    units=units,
    emb_dim=EMB_DIM,
    emb_similarity=rsa_loaded,
    compositional_on=False,
)

print(f'T_TaskEmbedding initialized')
print(f'  base_cols:    {emb.base_cols}')
print(f'  feature_cols: {emb.feature_cols}')
print(f'  total_emb_dim: {emb.total_emb_dim}  ({len(emb.base_cols)} base × {EMB_DIM} + RSRCONJ × {EMB_DIM})')

# Get embeddings for all 16 conditions
cond_idx = torch.arange(16)
with torch.no_grad():
    E = emb(cond_idx).numpy()   # (16, total_emb_dim)

print(f'\nEmbedding matrix shape: {E.shape}  (16 conditions × {E.shape[1]} dims)')

# Pairwise cosine similarity of full embedding
E_norm = E / np.linalg.norm(E, axis=1, keepdims=True)
sim_full = E_norm @ E_norm.T

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(sim_full, ax=ax, vmin=0, vmax=1, cmap='Blues',
            linewidths=0.3, linecolor='lightgray', square=True,
            xticklabels=False, yticklabels=False)
ax.set_title(f'Cosine similarity of T_TaskEmbedding\n(all features, emb_dim={EMB_DIM})', fontsize=10)
plt.tight_layout()
plt.show()

## Embedding Geometry — PCA

Project all 16 condition embeddings onto PC1–PC2.  
Color by RULE; marker by STIM.  

**Expected:** conditions sharing a rule (same color) cluster together — the RULE similarity  
from RSA initialization is preserved in the embedding space.

In [ ]:
pca = PCA(n_components=2)
E_pca = pca.fit_transform(E)   # (16, 2)

rule_labels = design['RULE'].values      # 1-4
stim_labels = design['STIM'].values      # 1-4
resp_labels = design['RESP'].values      # 1-4
rsrconj     = design['RSRCONJ'].values   # 1-16

colors  = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
markers = ['o', 's', '^', 'D']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: colored by RULE, marker by STIM
ax = axes[0]
for r in range(1, 5):
    for s in range(1, 5):
        mask = (rule_labels == r) & (stim_labels == s)
        ax.scatter(E_pca[mask, 0], E_pca[mask, 1],
                   color=colors[r - 1], marker=markers[s - 1],
                   s=120, zorder=3,
                   label=f'Rule {r}' if s == 1 else None)
        for idx in np.where(mask)[0]:
            ax.annotate(str(rsrconj[idx]),
                        (E_pca[idx, 0], E_pca[idx, 1]),
                        fontsize=8, ha='center', va='center',
                        color='white', fontweight='bold')

ax.set_title('PCA of T_TaskEmbedding\n(color=RULE, marker=STIM, label=RSRCONJ)',
             fontsize=10)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.legend(fontsize=9, title='Rule')

# Panel 2: heatmap of per-feature embedding cosine similarity
ax = axes[1]
feat_labels = (['RULE'] * EMB_DIM + ['STIM'] * EMB_DIM +
               ['RESP'] * EMB_DIM + ['RSRCONJ'] * EMB_DIM)
feat_colors = {'RULE': '#e41a1c', 'STIM': '#377eb8',
               'RESP': '#4daf4a', 'RSRCONJ': '#984ea3'}

sns.heatmap(sim_full, ax=ax, vmin=0, vmax=1, cmap='Blues',
            linewidths=0.2, linecolor='lightgray', square=True,
            xticklabels=range(1, 17), yticklabels=range(1, 17))
ax.set_title('Cosine similarity (full embedding)\nrows/cols = conditions 1-16', fontsize=10)
ax.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig(DIR_VIZ / 'embedding_geometry_4rules.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Variance explained: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, '
      f'PC2={pca.explained_variance_ratio_[1]*100:.1f}%')